# Data Merging — Stage 4 Assembly 04: Targets

## Input
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/agg_means_nan.parquet` (market-level, pre-clip/pre-fill)
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/panel_nan.parquet` (stock-level, pre-clip/pre-fill)
- `lib.config` — `OUT`, `CRASH_THRESH`

## Purpose
Builds the actual prediction targets — a continuous "5-day forward minimum return" and a binary "crash" indicator — for both the market-level (aggregate) and stock-level (panel) pipelines, then calibrates a Huber loss delta per split. Deliberately uses the `_nan` (pre-clip, float64) versions of the assembled tables rather than the model-ready ones, since targets must be built from real values, not from clipped/zero-filled features.

## Cell 1 — Load and Establish the Return Convention
Loads market dates/target returns and panel returns/caps, asserting the two tables share an identical date calendar.

**The core problem this cell solves:** raw CRSP `dlyret` at row `t` is the return *earned on* day `t` (close `t-1` → close `t`), so it's known at the close of day `t`, and a genuinely forward-looking target must be built from days `t+1` through `t+5`. But `target_daily_return` (from the aggregate pipeline) may already have been shifted forward somewhere upstream in Stage 2 — in which case the correct window would be `t` through `t+4` on that *already-shifted* series, describing the same five calendar days under a different indexing offset. Getting this wrong is a silent one-day information leak, so **the convention is verified empirically rather than assumed**: the notebook reconstructs a cap-weighted market return directly from the panel's `dlyret`/`dlycap`, and checks correlation against `target_daily_return` shifted by -1, 0, and +1. The best match is asserted to be lag `-1` (confirming `target_daily_return` is indeed pre-shifted); if this assertion ever fails, it signals the upstream convention changed and the downstream window logic needs adjusting before proceeding.

## Cell 2 — Build the Targets
Defines `minret_5d_pct = 100 × min(next 5 trading days' returns)` and `y_binary = minret_5d_pct < CRASH_THRESH`.

**Why raw percentage, not a z-score**, stated explicitly:
- A crash is a crash in absolute terms — z-scoring would divide out exactly the regime information a crash-prediction target needs to preserve.
- Avoids a subtle off-by-one trap: an expanding z-score looks *backward* from row `t`, while the target looks *forward* to `t+5`, so combining the two conventions would require an extra day of lag that's easy to get wrong silently.
- Keeps `y_binary` defined on the same raw scale as the continuous target, so both targets share one consistent unit.
- The `×100` is a pure rescale (no effect on correlation, R², or ranking) — purely to give the optimizer a more convenient numerical range.

**`forward_min()`** is the core helper, handling two different offset conventions via a `start` parameter:
- `start=1` (panel, raw `dlyret`): row `t` is the return realized *on* day `t`, so the forward window is `t+1..t+HORIZON`.
- `start=0` (market, `target_daily_return`): Stage 2 already shifted this series forward, so row `t` already holds day `t+1`, and the window is `t..t+HORIZON-1` — covering the *same* five calendar days as the panel case, just indexed differently. This is exactly the offset relationship verified empirically in Cell 1.

Two hard requirements enforced inside `forward_min`, both explicitly justified in the docstring:
- **Full window required.** `Series.min()` silently skips NaN by default, so without an explicit `.where(complete & ...)` guard, a row with only 2 of 5 forward returns available would quietly return a 2-day minimum instead of a 5-day one — a failure mode that would appear at every stock's tail and at the very end of the sample.
- **Contiguous window required.** `shift()` operates by row position, not by calendar date, so a stock that temporarily left the top-100 universe and later rejoined would get a "5-day forward window" that actually spans the gap between its two membership periods. This is checked against the actual shifted date via `MAX_WINDOW_DAYS = 14` (chosen because a genuine 5-trading-day window spans at most ~10 calendar days even across a holiday).

After construction:
- **Hand-verified spot checks** on both the market series (4 specific rows) and one panel stock (3 specific rows), recomputing the expected minimum directly from the raw return array and asserting agreement to `1e-9`.
- **Cross-pipeline consistency check.** Since the market target is pre-shifted and the panel target is not, both should nonetheless describe the *same* five calendar days on any shared date. This is verified by reconstructing a cap-weighted minimum from the panel's own `minret_5d_pct` values and correlating it against the market target — asserting `corr > 0.85`, with the explicit warning that a low correlation here would most likely indicate the two offset conventions are off by a day relative to each other.

## Cell 3 — Trim and Save
Both pipelines are trimmed to end on the same date: the last date with a *complete* forward window in either series (`min` of each series' own last valid date). Individual stocks' own tails are handled separately by the per-row completeness requirement already enforced in `forward_min` — a stock exiting the universe in 2015 loses only its own last 5 rows, not the global end-of-sample rows for every other stock.

Saves the final target tables and reports rows lost to incomplete windows, both in total and as median per-stock (expected to be ~5, matching the horizon).

## Cell 4 — Target Distribution Diagnostics
Reports mean/median/std/min/max/percentiles for both the market and panel `minret_5d_pct` series, plus crash rate (`y_binary` mean) for each.

Explicitly notes that the panel's crash rate is far higher than the market's, and explains *why* this is expected rather than a bug: an individual stock dropping 2% in a week is a routine occurrence, while the cap-weighted market as a whole rarely does. These are described as genuinely different prediction problems — the market target is a rare-event problem, the panel target is not — with implications for loss weighting and evaluation metric choice, but not for how the targets themselves were constructed.

Also reports crash rate by year (to inform split-boundary choices) and the 10 worst individual market windows by minimum return.

## Cell 5 — Huber Delta Calibration
Calibrates the Huber loss transition point `delta` — the value below which Huber behaves like MSE and above which an observation's gradient is capped and stops growing — separately for each of four splits (`Split_A`–`Split_D`), computed **only on each split's training window**, and separately for the market and panel targets.

Several design decisions are spelled out explicitly:

- **Why delta matters here specifically:** the panel target reaches as low as -94.2% (identified in comments as the Washington Mutual collapse) — under a pure MSE loss, that single observation would outweigh roughly a thousand ordinary ones.
- **Delta is a multiple of a robust scale, not a fixed number.** On the old z-scored target, `delta=3` meant "3 standard deviations" because z-scores have unit sd by construction. Keeping a literal value of 3 on *raw percentage* targets would silently correspond to different numbers of standard deviations for market vs. panel (2.4σ vs. 1.5σ respectively) — giving the two pipelines inconsistent robustness. The actual free parameter has always been the multiple (`DELTA_ROBUST_SIGMA = 3.0`), not a raw number.
- **Per-split, training-window-only.** Computing delta from the full sample (including test-period observations) would mean test-period target values are configuring the training loss function — described as a *stronger* form of leakage than the label-free leakage this whole pipeline otherwise guards against elsewhere.

**`huber_delta()`** computes `delta = DELTA_ROBUST_SIGMA × 1.4826 × MAD(y_train)`:
- **MAD rather than sample std**, because sample std is itself inflated by exactly the outliers delta is meant to bound — e.g. the panel target has sample sd 2.04 against a robust MAD-based scale of ~1.23, so `3 × sample_sd` would sit *beyond* the actual data and Huber would degenerate back into plain MSE. The docstring notes the actual failure mode this guards against is delta being too *loose* for ordinary observations, not outliers escaping — an outlier that inflates sample sd inflates its own position *in sd-units* by even more, so it structurally cannot hide inside a 3-sigma-in-MAD-units boundary.
- **Set from the target's own dispersion, even though Huber technically acts on residuals.** Residual sd = target sd × √(1 − R²), so using the raw target's dispersion as a proxy systematically overstates the correct residual-based delta by an amount depending on model predictability. No correction is applied, with the stated reasoning that out-of-sample R² varies from ~0.35 down to negative across different splits and market regimes, so no single correction constant would be defensible — and at that scale of uncertainty, the correction would be smaller than the judgment call of choosing 2.5 vs. 3.0 for the multiple in the first place.
- **Uses plain MAD, not the project's `robust_scale_ladder`** (referenced elsewhere in the pipeline for features with heavily tied values) — a continuous return series cannot produce the kind of >50%-tied-value degeneracy that ladder exists to handle, so if MAD ever does collapse near zero here, that's treated as a genuine data error to investigate (enforced via an assertion), not something to gracefully fall back from.

Reports a full diagnostic table per split/target combination (`inflation` = sample sd / robust sd, `delta_in_sd` = delta expressed in sample-sd units, `pct_beyond` = fraction of training targets beyond delta from the median — explicitly flagged as "a guide only," since Huber acts on residuals, not raw targets) and a `spread` metric showing how much delta varies across splits for each target.

**Panel decomposition (a side analysis, not used to set delta):** decomposes the panel target into a common market component (per-date mean) and an idiosyncratic residual, computing the robust scale of each. This quantifies how much genuinely stock-specific variation exists for the panel pipeline to potentially capture that the aggregate pipeline structurally cannot — framed as a result in its own right about the two pipelines' relative information content, and used to bound how "loose" delta would become in residual terms in the best case (if the market component were perfectly predicted) versus the realistic case (given the aggregate's own R² of 0.35 to negative).

## Output
- `Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets/targets_market.parquet` — `date`, `target_daily_return`, `minret_5d_pct`, `y_binary`.
- `Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets/targets_panel.parquet` — `permno`, `date`, `dlyret`, `minret_5d_pct`, `y_binary`.
- `Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets/huber_delta.csv` — per-split, per-target delta calibration table.
- `Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets/huber_delta.json` — same calibration, machine-readable, including the panel decomposition (`panel_pooled_mad`, `panel_idio_mad`) for downstream reuse.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.config import OUT, CRASH_THRESH

ASM_DIR = OUT / '02_assembled'
OUT_DIR = OUT / '03_targets'
OUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 5
MAX_WINDOW_DAYS = 14   # 5 trading days spans <= ~10 calendar days even over
                       # a holiday. Anything longer means the window jumped a
                       # coverage gap -- a stock that left the universe and
                       # returned -- and is not a real 5-day window.

pd.set_option('display.width', 200)

print('=' * 100)
print('LOAD RETURNS')
print('=' * 100)

# _nan versions: float64 and pre-cast. agg_means.parquet is float32.
mkt = (pd.read_parquet(ASM_DIR / 'agg_means_nan.parquet',
                       columns=['date', 'target_daily_return'])
         .sort_values('date').reset_index(drop=True))
pan = (pd.read_parquet(ASM_DIR / 'panel_nan.parquet',
                       columns=['permno', 'date', 'dlyret', 'dlycap'])
         .sort_values(['permno', 'date'], kind='stable').reset_index(drop=True))

print(f'  market  {len(mkt):>7,} dates   {mkt["date"].min().date()} .. {mkt["date"].max().date()}')
print(f'  panel   {len(pan):>7,} rows    {pan["permno"].nunique()} permnos')
assert set(mkt['date']) == set(pan['date']), 'market and panel calendars differ'

# ── WHICH DAY DOES target_daily_return DESCRIBE? ─────────────────────────────
# CRSP dlyret at date t is the return earned ON day t (close t-1 -> close t),
# so it is known at the close of day t. The features at row t are also known at
# the close of day t. A forward target must therefore use t+1..t+5.
#
# But `target_daily_return` may already have been shifted forward somewhere in
# Stage 2, in which case t..t+4 would be right. Getting this wrong is a silent
# one-day leak, so it is established from the data rather than assumed:
# reconstruct the cap-weighted market return from the panel and see which lag
# matches.

print('\n' + '=' * 100)
print('RETURN CONVENTION CHECK')
print('=' * 100)

cw = (pan.groupby('date')
         .apply(lambda g: np.average(g['dlyret'], weights=g['dlycap']),
                include_groups=False)
         .rename('cw_ret').reset_index())
chk = mkt.merge(cw, on='date', how='inner')

for lag in (-1, 0, 1):
    r = chk['target_daily_return'].corr(chk['cw_ret'].shift(lag))
    note = {0:  'target_daily_return[t] = return realised ON day t',
            -1: 'target_daily_return[t] = return on day t+1 (already forward)',
            1:  'target_daily_return[t] = return on day t-1 (lagged)'}[lag]
    print(f'  corr with cw_ret.shift({lag:>2}) = {r:.6f}   {note}')

best = max((-1, 0, 1), key=lambda l: abs(chk['target_daily_return']
                                         .corr(chk['cw_ret'].shift(l))))
assert best == -1, (
    f'target_daily_return best matches lag {best}, not -1. Stage 2 shifted this '
    f'series forward, so row t already holds day t+1; the window below is t..t+4 '
    f'on the shifted series. If the convention has changed, adjust before running.')
print(f'\n  -> already forward-shifted in Stage 2. Window is t .. t+{HORIZON-1} '
      f'on this series, i.e. calendar days t+1 .. t+{HORIZON}.')

LOAD RETURNS
  market    4,384 dates   2007-08-01 .. 2024-12-30
  panel   436,669 rows    208 permnos

RETURN CONVENTION CHECK
  corr with cw_ret.shift(-1) = 0.999704   target_daily_return[t] = return on day t+1 (already forward)
  corr with cw_ret.shift( 0) = -0.135718   target_daily_return[t] = return realised ON day t
  corr with cw_ret.shift( 1) = -0.005504   target_daily_return[t] = return on day t-1 (lagged)

  -> already forward-shifted in Stage 2. Window is t .. t+4 on this series, i.e. calendar days t+1 .. t+5.


In [2]:
# ── THE TARGET ───────────────────────────────────────────────────────────────
#   minret_5d_pct = 100 * min of the next 5 trading days' returns
#   y_binary      = minret_5d_pct < CRASH_THRESH
#
# Raw percentage, not an expanding z-score. Three reasons:
#   - a crash is a crash in absolute terms; z-scoring divides out exactly the
#     regime information a crash model should be using
#   - no shift=6 trap (an expanding z looks back from t while the target looks
#     forward to t+5, so the stats need an extra day of lag; getting it wrong
#     is a silent leak)
#   - y_binary is defined on raw returns, so both targets share one scale
#
# x100 is a pure rescale: no correlation, R^2 or ranking changes, only the
# numerical range the optimiser sees.

def forward_min(ret, dates, group=None, start=1):
    """
    100 * min over HORIZON returns, with two hard requirements.

    start=1  raw CRSP dlyret -- row t is the return realised ON day t, so the
             forward window is t+1 .. t+HORIZON.
    start=0  target_daily_return -- Stage 2 already shifted this series forward,
             so row t holds day t+1 and the window is t .. t+HORIZON-1. The same
             five calendar days either way; the offset just differs. Verified
             empirically in cell 1 (corr 0.9997 at lag -1 vs -0.14 at lag 0).

    FULL WINDOW REQUIRED. Series.min() skips NaN, so without the .where() a row
    with only 2 of 5 forward returns would silently return a 2-day minimum --
    at every stock's tail and at the end of the sample.

    CONTIGUOUS WINDOW REQUIRED. shift() pairs by row position, not by date, so a
    stock that left the top-100 universe and later rejoined would get a "5-day
    forward window" spanning the gap. Checked against the shifted date.

    Returns (target, complete_mask, contiguous_mask).
    """
    offs = range(start, start + HORIZON)
    last = start + HORIZON - 1          # offset of the final day in the window

    if group is None:
        fwd  = pd.concat([ret.shift(-h) for h in offs], axis=1)
        dfwd = dates.shift(-last)
    else:
        g, gd = ret.groupby(group, sort=False), dates.groupby(group, sort=False)
        fwd  = pd.concat([g.shift(-h) for h in offs], axis=1)
        dfwd = gd.shift(-last)

    complete   = fwd.notna().all(axis=1)
    contiguous = (dfwd - dates).dt.days.le(MAX_WINDOW_DAYS)
    return (fwd.min(axis=1) * 100).where(complete & contiguous), complete, contiguous


print('=' * 100)
print('BUILD TARGETS')
print('=' * 100)

# ── market: start=0, the series is pre-shifted ──
m, m_complete, m_contig = forward_min(mkt['target_daily_return'], mkt['date'],
                                      start=0)
mkt['minret_5d_pct'] = m
mkt['y_binary'] = np.where(mkt['minret_5d_pct'].isna(), np.nan,
                           (mkt['minret_5d_pct'] < CRASH_THRESH).astype(float))
print(f'  market   {int(m.notna().sum()):>7,} of {len(mkt):>7,} rows have a target '
      f'({int((~m_complete).sum())} incomplete, '
      f'{int((m_complete & ~m_contig).sum())} non-contiguous)')

# ── panel: start=1, raw CRSP dlyret ──
p, p_complete, p_contig = forward_min(pan['dlyret'], pan['date'],
                                      group=pan['permno'], start=1)
pan['minret_5d_pct'] = p
pan['y_binary'] = np.where(pan['minret_5d_pct'].isna(), np.nan,
                           (pan['minret_5d_pct'] < CRASH_THRESH).astype(float))
print(f'  panel    {int(p.notna().sum()):>7,} of {len(pan):>7,} rows have a target '
      f'({int((~p_complete).sum()):,} incomplete, '
      f'{int((p_complete & ~p_contig).sum()):,} non-contiguous)')

# ── spot-checks, computed by hand from the raw arrays ────────────────────────
r = mkt['target_daily_return'].to_numpy()
for i in (0, 500, 2000, len(mkt) - HORIZON - 1):
    expected = 100 * np.min(r[i: i + HORIZON])          # start=0
    actual   = mkt['minret_5d_pct'].iloc[i]
    assert abs(expected - actual) < 1e-9, \
        f'market spot-check failed at row {i}: {expected} vs {actual}'
print('  spot-checks passed (market, 4 rows verified by hand)')

one = pan[pan['permno'] == pan['permno'].iloc[0]].reset_index(drop=True)
r1 = one['dlyret'].to_numpy()
for i in (0, 100, len(one) - HORIZON - 1):
    expected = 100 * np.min(r1[i + 1: i + 1 + HORIZON])  # start=1
    actual   = one['minret_5d_pct'].iloc[i]
    assert abs(expected - actual) < 1e-9, \
        f'panel spot-check failed at row {i}: {expected} vs {actual}'
print('  spot-checks passed (panel, one stock, 3 rows verified by hand)')

# ── the two targets must describe the same five calendar days ────────────────
# The market series is pre-shifted and the panel is not, so on any shared date
# they should agree on WHICH days the window covers. Reconstructing the
# cap-weighted minimum from the panel and comparing to the market target checks
# the two offsets are consistent -- a one-day error would show as a weak match.

cw_min = (pan.dropna(subset=['minret_5d_pct'])
             .groupby('date')
             .apply(lambda g: np.average(g['minret_5d_pct'], weights=g['dlycap']),
                    include_groups=False)
             .rename('cw_minret').reset_index())
align = mkt.dropna(subset=['minret_5d_pct']).merge(cw_min, on='date', how='inner')
corr = align['minret_5d_pct'].corr(align['cw_minret'])
print(f'\n  market target vs cap-weighted panel minimum: corr {corr:.4f} '
      f'over {len(align):,} shared dates')
assert corr > 0.85, (
    f'corr {corr:.4f} is too low -- the market and panel windows are probably '
    f'offset by a day. Re-check the start= arguments above.')

BUILD TARGETS
  market     4,380 of   4,384 rows have a target (4 incomplete, 0 non-contiguous)
  panel    435,329 of 436,669 rows have a target (1,040 incomplete, 300 non-contiguous)
  spot-checks passed (market, 4 rows verified by hand)
  spot-checks passed (panel, one stock, 3 rows verified by hand)

  market target vs cap-weighted panel minimum: corr 0.9614 over 4,379 shared dates


In [3]:
# Both pipelines end on the same date: the last date with a complete forward
# window. Per-stock tails are separate and already handled by the full-window
# requirement -- a stock that exits in 2015 loses its own last 5 rows, not the
# global ones.

print('=' * 100)
print('TRIM AND SAVE')
print('=' * 100)

last_valid = min(mkt.loc[mkt['minret_5d_pct'].notna(), 'date'].max(),
                 pan.loc[pan['minret_5d_pct'].notna(), 'date'].max())
print(f'  last date with a complete forward window: {last_valid.date()}')

tm = mkt[mkt['date'] <= last_valid].dropna(subset=['minret_5d_pct']).reset_index(drop=True)
tp = pan[pan['date'] <= last_valid].dropna(subset=['minret_5d_pct']).reset_index(drop=True)

tm[['date', 'target_daily_return', 'minret_5d_pct', 'y_binary']].to_parquet(
    OUT_DIR / 'targets_market.parquet', index=False)
tp[['permno', 'date', 'dlyret', 'minret_5d_pct', 'y_binary']].to_parquet(
    OUT_DIR / 'targets_panel.parquet', index=False)

print(f'  targets_market.parquet   {len(tm):>7,} rows   '
      f'{tm["date"].min().date()} .. {tm["date"].max().date()}')
print(f'  targets_panel.parquet    {len(tp):>7,} rows   '
      f'{tp["permno"].nunique()} permnos   {tp["date"].nunique():,} dates')

# Rows lost per stock at its own tail, not the global one.
tail_lost = (pan.groupby('permno').size() - tp.groupby('permno').size()).fillna(0)
print(f'\n  rows lost to incomplete windows: {len(pan) - len(tp):,} total, '
      f'median {tail_lost.median():.0f} per stock (expect ~5)')

TRIM AND SAVE
  last date with a complete forward window: 2024-12-20
  targets_market.parquet     4,379 rows   2007-08-01 .. 2024-12-20
  targets_panel.parquet    435,329 rows   208 permnos   4,379 dates

  rows lost to incomplete windows: 1,340 total, median 5 per stock (expect ~5)


In [4]:
print('=' * 100)
print('TARGET DISTRIBUTIONS')
print('=' * 100)

for name, s in [('market', tm['minret_5d_pct']), ('panel', tp['minret_5d_pct'])]:
    print(f'\n  {name}')
    print(f'    mean {s.mean():>8.3f}%   median {s.median():>8.3f}%   '
          f'std {s.std():>7.3f}')
    print(f'    min  {s.min():>8.3f}%   max    {s.max():>8.3f}%')
    print('    percentiles: ' + '  '.join(
        f'p{int(q*100)}={s.quantile(q):.2f}' for q in (.01, .05, .25, .50, .75, .95, .99)))

print('\n' + '-' * 100)
print(f'CRASH RATE   (minret_5d_pct < {CRASH_THRESH})')
print('-' * 100)
print(f'  market   {tm["y_binary"].mean():>7.2%}  ({int(tm["y_binary"].sum()):>6,} '
      f'of {len(tm):>7,})')
print(f'  panel    {tp["y_binary"].mean():>7.2%}  ({int(tp["y_binary"].sum()):>6,} '
      f'of {len(tp):>7,})')
print('\n  The panel rate is far higher because an individual stock drops 2% in a')
print('  week routinely, while the cap-weighted market rarely does. These are')
print('  different problems: the market target is a rare event, the panel target')
print('  is not. Affects loss weighting and metric choice, not the construction.')

print('\n' + '-' * 100)
print('CRASH RATE BY YEAR   (informs split choice)')
print('-' * 100)
by_year = pd.DataFrame({
    'market': tm.groupby(tm['date'].dt.year)['y_binary'].mean(),
    'panel':  tp.groupby(tp['date'].dt.year)['y_binary'].mean(),
    'mkt_worst': tm.groupby(tm['date'].dt.year)['minret_5d_pct'].min(),
})
print(by_year.to_string(formatters={'market': '{:.1%}'.format,
                                    'panel':  '{:.1%}'.format,
                                    'mkt_worst': '{:.2f}%'.format}))

print('\n' + '-' * 100)
print('WORST 10 MARKET WINDOWS')
print('-' * 100)
print(tm.nsmallest(10, 'minret_5d_pct')[['date', 'minret_5d_pct']]
        .to_string(index=False, formatters={'minret_5d_pct': '{:.2f}%'.format}))

TARGET DISTRIBUTIONS

  market
    mean   -1.176%   median   -0.834%   std   1.232
    min   -11.774%   max       0.886%
    percentiles: p1=-5.60  p5=-3.29  p25=-1.62  p50=-0.83  p75=-0.34  p95=-0.02  p99=0.14

  panel
    mean   -1.873%   median   -1.378%   std   2.039
    min   -94.247%   max      13.954%
    percentiles: p1=-9.53  p5=-5.15  p25=-2.39  p50=-1.38  p75=-0.73  p95=-0.10  p99=0.32

----------------------------------------------------------------------------------------------------
CRASH RATE   (minret_5d_pct < -2.0)
----------------------------------------------------------------------------------------------------
  market    17.97%  (   787 of   4,379)
  panel     32.73%  (142,501 of 435,329)

  The panel rate is far higher because an individual stock drops 2% in a
  week routinely, while the cap-weighted market rarely does. These are
  different problems: the market target is a rare event, the panel target
  is not. Affects loss weighting and metric choice, not the c

In [5]:
mkt_by_date = tp.groupby('date')['minret_5d_pct'].transform('mean')
idio = tp['minret_5d_pct'] - mkt_by_date
print(f'pooled MAD : {1.4826 * (tp["minret_5d_pct"] - tp["minret_5d_pct"].median()).abs().median():.3f}')
print(f'idio MAD   : {1.4826 * (idio - idio.median()).abs().median():.3f}')

pooled MAD : 1.138
idio MAD   : 0.858


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# HUBER DELTA
# ═══════════════════════════════════════════════════════════════════════════════
# delta is the crossover between quadratic and linear. Below it Huber behaves as
# MSE; above it an observation's gradient is capped at delta and stops growing.
# The panel target reaches -94.2% (Washington Mutual); under pure MSE that one
# row would outweigh a thousand ordinary observations.
#
# A MULTIPLE, NOT A NUMBER. delta=3 on the old z-scored target meant "3 standard
# deviations", because z-scores have sd ~ 1. Keeping the literal 3 on raw
# percentages silently makes it 2.4 sigma (market) or 1.5 sigma (panel), giving
# the two pipelines different robustness. The multiple was always the decision.
#
# PER SPLIT, TRAINING WINDOW ONLY. delta is computed from y, so a full-sample
# delta means test-period target values configured the training loss -- stronger
# than the label-free leakage guarded against elsewhere in this pipeline.

import json

DELTA_ROBUST_SIGMA = 3.0

TRAIN_END = {
    'Split_A': '2015-12-31',
    'Split_B': '2017-12-31',
    'Split_C': '2019-12-31',
    'Split_D': '2020-12-31',
}


def huber_delta(y, label):
    """
    delta = DELTA_ROBUST_SIGMA * 1.4826 * MAD(y_train)

    MAD rather than sd because sd is inflated by the outliers delta exists to
    bound -- the panel target has sd 2.04 against a robust scale near 1.23, so
    3*sd would sit beyond the data and Huber would degenerate into MSE. Note the
    failure mode is delta too LOOSE for ordinary observations, not outliers
    escaping: an outlier that inflates sd inflates its own position in sd-units
    by more, so it can never hide inside 3 sigma.

    Set from the TARGET's dispersion, though Huber acts on RESIDUALS. Residual
    sd = target sd * sqrt(1 - R2), so the proxy overstates by an amount that
    depends on predictability. No correction is applied: out-of-sample R2 varies
    from ~0.35 to negative across splits and regimes, so no single constant is
    defensible, and at these levels the adjustment is smaller than the choice
    between a multiple of 2.5 and 3.

    Plain MAD, not robust_scale_ladder -- the ladder's rungs 2-4 handle features
    with >50% tied values. A continuous return series cannot hit them, and a
    collapsed MAD is a data error to investigate, not to handle gracefully.
    """
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]

    med   = float(np.median(y))
    mad   = float(np.median(np.abs(y - med)))
    sigma = 1.4826 * mad
    assert sigma > 1e-8, (
        f'{label}: MAD is ~zero, target near-constant. Investigate.\n'
        f'{pd.Series(y).describe()}')

    delta = DELTA_ROBUST_SIGMA * sigma
    return {
        'n_train':      int(len(y)),
        'median':       med,
        'robust_sigma': sigma,
        'sample_sigma': float(np.std(y, ddof=1)),
        'inflation':    float(np.std(y, ddof=1) / sigma),
        'delta':        float(delta),
        'delta_in_sd':  float(delta / np.std(y, ddof=1)),
        'pct_beyond':   float((np.abs(y - med) > delta).mean()),
    }


# ── RUN ──────────────────────────────────────────────────────────────────────

print('=' * 104)
print('HUBER DELTA  (per split, training window only)')
print('=' * 104)
print(f'  delta = {DELTA_ROBUST_SIGMA} x 1.4826 x MAD(y_train)\n')

rows = []
for split, end in TRAIN_END.items():
    for name, df in [('market', tm), ('panel', tp)]:
        y = df.loc[df['date'] <= end, 'minret_5d_pct']
        rows.append({'split': split, 'target': name, 'train_end': end,
                     **huber_delta(y, f'{split}/{name}')})

deltas = pd.DataFrame(rows)

print(deltas[['split', 'target', 'train_end', 'n_train', 'robust_sigma',
              'sample_sigma', 'inflation', 'delta', 'delta_in_sd', 'pct_beyond']]
      .to_string(index=False, formatters={
          'n_train':      '{:,}'.format,
          'robust_sigma': '{:.3f}'.format,
          'sample_sigma': '{:.3f}'.format,
          'inflation':    '{:.2f}x'.format,
          'delta':        '{:.3f}'.format,
          'delta_in_sd':  '{:.2f}'.format,
          'pct_beyond':   '{:.2%}'.format,
      }))

print('\n' + '-' * 104)
print('THE DELTAS')
print('-' * 104)
for name, g in deltas.groupby('target'):
    vals = '   '.join(f'{r["split"].replace("Split_", "")}={r["delta"]:.3f}'
                      for _, r in g.iterrows())
    lo, hi = g['delta'].min(), g['delta'].max()
    print(f'  {name:<8} {vals}      spread {100*(hi-lo)/lo:.1f}%')

print('\n' + '-' * 104)
print('READING THE TABLE')
print('-' * 104)
print('  inflation    sample sd / robust sd. Above ~1.3 means the tail is')
print('               distorting sd -- this number IS the case for using MAD.')
print('  delta_in_sd  delta in sample-sd units. Below 3.0 by exactly the')
print('               inflation factor; that gap is the correction working.')
print('  pct_beyond   fraction of training TARGETS beyond delta from the median.')
print('               A guide only -- Huber acts on residuals, not targets.')
print('  spread       variation in delta across splits. Above ~15% would mean')
print('               the loss differs materially between splits.')

# ── PANEL DECOMPOSITION ──────────────────────────────────────────────────────
# Not used to set delta, but it bounds how loose delta is in residual terms and
# is a result in its own right: it quantifies how much stock-specific variation
# there is for the panel pipeline to find that the aggregate cannot.

mkt_comp = tp.groupby('date')['minret_5d_pct'].transform('mean')
idio     = tp['minret_5d_pct'] - mkt_comp
pooled_mad = 1.4826 * float((tp['minret_5d_pct'] - tp['minret_5d_pct'].median()).abs().median())
idio_mad   = 1.4826 * float((idio - idio.median()).abs().median())

print('\n' + '-' * 104)
print('PANEL TARGET DECOMPOSITION')
print('-' * 104)
print(f'  pooled robust scale        {pooled_mad:.3f}')
print(f'  idiosyncratic robust scale {idio_mad:.3f}   ({idio_mad/pooled_mad:.0%} of pooled)')
print(f'  common market component    {1 - idio_mad/pooled_mad:.0%}')
print()
print('  If the model captured the market factor perfectly, residuals would')
print('  shrink to the idiosyncratic scale and delta would be ~33% loose in')
print('  residual terms. It will not -- predicting the market weekly low is the')
print('  aggregate problem, R2 0.35 to negative -- so residuals land between the')
print('  two and delta is roughly 3.0-4.0 residual sigmas.')

# ── SAVE ─────────────────────────────────────────────────────────────────────

deltas.to_csv(OUT_DIR / 'huber_delta.csv', index=False)

with open(OUT_DIR / 'huber_delta.json', 'w') as f:
    json.dump({
        'rule':                'delta = DELTA_ROBUST_SIGMA * 1.4826 * MAD(y_train)',
        'delta_robust_sigma':  DELTA_ROBUST_SIGMA,
        'scope':               'per split, training window only',
        'panel_pooled_mad':    pooled_mad,
        'panel_idio_mad':      idio_mad,
        'deltas': {f'{r["split"]}/{r["target"]}': round(r['delta'], 4)
                   for r in deltas.to_dict('records')},
    }, f, indent=2)

print(f'\n  saved -> huber_delta.csv, huber_delta.json')

HUBER DELTA  (per split, training window only)
  delta = 3.0 x 1.4826 x MAD(y_train)

  split target  train_end n_train robust_sigma sample_sigma inflation delta delta_in_sd pct_beyond
Split_A market 2015-12-31   2,121        0.957        1.303     1.36x 2.871        2.20      4.34%
Split_A  panel 2015-12-31 210,747        1.169        2.287     1.96x 3.507        1.53      6.52%
Split_B market 2017-12-31   2,624        0.825        1.233     1.50x 2.474        2.01      4.61%
Split_B  panel 2017-12-31 260,441        1.102        2.151     1.95x 3.305        1.54      6.55%
Split_C market 2019-12-31   3,127        0.808        1.196     1.48x 2.423        2.03      5.21%
Split_C  panel 2019-12-31 310,289        1.095        2.065     1.89x 3.286        1.59      6.22%
Split_D market 2020-12-31   3,380        0.833        1.306     1.57x 2.500        1.91      5.47%
Split_D  panel 2020-12-31 335,544        1.129        2.127     1.88x 3.387        1.59      6.32%

----------------------